# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 원본 raw 엑셀(`data/raw_q.xlsx`)과 중간 산출 CSV(`output/crashp(6m,35%).csv`, `output/jackp(6m,35%).csv`)은(는) 비공개이거나 용량이 커서 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

일부 셀은 벤치마크 지수를 `yfinance`로 내려받으므로 인터넷이 필요합니다. 확률 저장 셀은 crash/jump 중 하나만 번갈아 저장하도록 주석 처리되어 있어, 두 CSV를 모두 얻으려면 예측 인덱스(`[1]` crash, `[2]` jump)를 바꿔 두 번 실행해야 합니다.

In [ ]:
import copy
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix

In [ ]:
def calculate_quantile_df(df, num_quantiles):
    """
    df: 분위수를 계산할 데이터프레임
    num_quantiles: 생성할 분위수 개수
    """
    quantiles = [i / num_quantiles for i in range(num_quantiles + 1)]
    quantile_dfs = []

    for i in range(num_quantiles):
        lower_bound = quantiles[i]
        upper_bound = quantiles[i + 1]

        # 행 기준 분위수 계산 (axis=1), 열 기준이면 axis=0
        lower_threshold = df.quantile(lower_bound, axis=1)
        upper_threshold = df.quantile(upper_bound, axis=1)

        # 마지막 분위수(i == num_quantiles-1)만 upper_threshold를 포함(<=)하도록 처리
        if i < num_quantiles - 1:
            quantile_df = df.apply(
                lambda row: pd.Series(
                    np.where(
                        (row >= lower_threshold[row.name]) & (row < upper_threshold[row.name]),
                        row,
                        np.nan
                    ),
                    index=row.index
                ),
                axis=1
            )
        else:
            # 가장 큰 분위수에 대해서는 upper_threshold 포함
            quantile_df = df.apply(
                lambda row: pd.Series(
                    np.where(
                        (row >= lower_threshold[row.name]) & (row <= upper_threshold[row.name]),
                        row,
                        np.nan
                    ),
                    index=row.index
                ),
                axis=1
            )

        quantile_dfs.append(quantile_df)

    return quantile_dfs


from scipy.stats.mstats import winsorize
# 데이터프레임 전체에 윈저라이징 적용
def winsorize_dataframe_entire(df, lower_percentile=0.0, upper_percentile=0.05):
    """
    데이터프레임 전체에 대해 윈저라이징을 적용.
    - lower_percentile: 하한 퍼센타일
    - upper_percentile: 상한 퍼센타일
    """
    # 데이터프레임을 1차원 배열로 변환
    flattened = df.values.flatten()
    
    # 윈저라이징 적용
    winsorized = winsorize(flattened, limits=(lower_percentile, upper_percentile))
    
    # 윈저라이징된 데이터를 원래 데이터프레임 형태로 복원
    return pd.DataFrame(winsorized.reshape(df.shape), columns=df.columns, index=df.index)

# 분위수 포트폴리오 생성
def calculate_quantile_ranges(df, return_df, num_quantiles):
    
    """
    df: 조건 데이터프레임
    return_df: 수익률 데이터프레임
    num_quantiles: 생성할 분위수 개수
    """
    
    quantiles = [i / num_quantiles for i in range(num_quantiles + 1)]
    quantile_dfs = []
    quantile_portfolio = []
    
    for i in range(num_quantiles):
        lower_bound = quantiles[i]
        upper_bound = quantiles[i + 1]
        
        # 행 단위 분위수 계산 (axis=1)
        lower_threshold = df.quantile(lower_bound, axis=1)
        upper_threshold = df.quantile(upper_bound, axis=1)
        
        # 마지막 분위수 구간에서는 '==' upper_threshold 값도 포함
        if i < num_quantiles - 1:
            quantile_df = df.apply(
                lambda row: pd.Series(
                    np.where(
                        (row >= lower_threshold[row.name]) & (row < upper_threshold[row.name]),
                        row,
                        np.nan
                    ),
                    index=row.index
                ),
                axis=1
            )
        else:
            quantile_df = df.apply(
                lambda row: pd.Series(
                    np.where(
                        (row >= lower_threshold[row.name]) & (row <= upper_threshold[row.name]),
                        row,
                        np.nan
                    ),
                    index=row.index
                ),
                axis=1
            )
        
        quantile_dfs.append(quantile_df)
    
    for i in range(len(quantile_dfs)):
        condition = quantile_dfs[i].applymap(lambda x: 1 if not pd.isna(x) else np.nan)
        quantile_portfolio.append((return_df * condition).dropna(how='all'))
    
    return quantile_portfolio  # 분위수별로 작은 순서대로 반환


In [ ]:
# 엑셀 파일의 모든 시트를 읽는 함수
def read_sheet(sheet_name, file_path):
    """특정 시트를 읽어서 반환합니다."""
    return sheet_name, pd.read_excel(file_path, sheet_name=sheet_name, index_col=0, parse_dates=True, header=8)

# 병렬 처리를 위한 함수
def read_excel_parallel(file_path, num_jobs=-1):
    """
    병렬로 엑셀 파일의 모든 시트를 읽어옵니다.
    :param file_path: 엑셀 파일 경로
    :param num_jobs: 병렬 처리에 사용할 작업 수 (-1은 모든 CPU 코어 사용)
    :return: 시트 이름을 키로 하고 DataFrame을 값으로 가지는 딕셔너리
    """
    # 엑셀 파일에서 시트 목록 가져오기
    sheet_names = pd.ExcelFile(file_path).sheet_names

    # 병렬 처리로 시트 읽기
    results = Parallel(n_jobs=num_jobs)(
        delayed(read_sheet)(sheet_name, file_path) for sheet_name in sheet_names
    )

    # 결과를 딕셔너리로 변환
    return {sheet: data for sheet, data in results}

# 사용 예제
file_path = 'data/raw_q.xlsx'
data = read_excel_parallel(file_path)

# 특정 시트 확인
print(data.keys())  # 읽어온 시트 이름들


# Raw Data

In [ ]:
price0 = data['수정주가'].iloc[5:]
price0.index = pd.to_datetime(price0.index)
price0 = price0.astype(float)

num_share = data['상장주식수'].iloc[5:]
num_share.index = pd.to_datetime(num_share.index)
num_share = num_share.astype(float)

volume = data['거래량'].iloc[5:]
volume.index = pd.to_datetime(volume.index)
volume = volume.astype(float)

tangible = data['유형자산'].iloc[5:]
tangible.index = pd.to_datetime(tangible.index)
tangible = tangible.astype(float)

sales = data['매출액(TTM)'].iloc[5:]
sales.index = pd.to_datetime(sales.index)
sales = sales.astype(float)

toa = data['총자산'].iloc[5:]
toa.index = pd.to_datetime(toa.index)
toa = toa.astype(float)

retail = data['개인순매수수량'].iloc[5:]
retail.index = pd.to_datetime(retail.index)
retail = retail.astype(float)

# init = data['기관순매수수량'].iloc[5:]
# init.index = pd.to_datetime(init.index)
# init = init.astype(float)

kospi = data['코스피'].iloc[5:][['코스피']]
kospi.index = pd.to_datetime(kospi.index)
kospi = kospi.astype(float)

capital = data['자본금'].iloc[5:]
capital.index = pd.to_datetime(capital.index)
capital = capital.astype(float)

toe = data['자본총계'].iloc[5:]
toe.index = pd.to_datetime(toe.index)
toe = toe.astype(float)

# 전처리
- 길이 50안되는 종목 제외

In [ ]:
# 180일 안되는 종목 삭제 (여유롭게 200일)
price = price0.loc[:, price0.count() > 365 * 1.5]

In [ ]:
financial_keywords = ['은행', '증권', '캐피탈', '보험', '카드', '자산', '운용', '신탁', '저축', '금고', '신용', '스팩', '금융','화재']
price = price.loc[:, ~price.columns.str.contains('|'.join(financial_keywords))]


In [ ]:
ticker = price.columns
print(len(ticker))

In [ ]:
num_share = num_share[ticker]
volume = volume[ticker]
tangible = tangible[ticker]
sales = sales[ticker]
retail = retail[ticker]
# init = init[ticker]
toa = toa[ticker]
capital = capital[ticker]
toe = toe[ticker]

# 변수 엔지니어링

## 수익률

In [ ]:
ret_daily = price.pct_change(fill_method=None).dropna(how='all')
ret = price.resample('ME').last().pct_change(fill_method=None).dropna(how='all')
ret_q = price.resample('QE').last().pct_change(fill_method=None).dropna(how='all')
ret_6m = price.resample('ME').last().pct_change(fill_method=None, periods=5).dropna(how='all')

## RM5
Measured at the end of month t as the log return
of the CRSP value-weighted index from month t − 5
to month t

In [ ]:
RM12 = np.log1p(kospi.resample('ME').last().pct_change(periods=5).dropna())

## EXRET12
Measured for each stock at the end of month
t as the log return on the stock from month t − 11 to
month t, in excess of the same return on the CRSP
value-weighted index (RM12)

In [ ]:
EXRET12 = (np.log1p(ret_6m)).sub(RM12['코스피'], axis=0).dropna(how='all')

## TVOL
Total volatility at the end of month t is the standard deviation of daily log returns from month t − 5
to month t. We exclude stocks with fewer than 50
daily log returns during the six-month period.

In [ ]:
TVOL = (np.log1p(ret_daily)).rolling(25*5).std().resample('ME').last()

## TSKEW
Total skewness at the end of month t is calculated using daily log returns from month t − 5 to
month t. We exclude stocks with fewer than 50 daily
log returns during the six-month period.

In [ ]:
TSKEW = (np.log1p(ret_daily)).rolling(25*5).skew().resample('ME').last()

## SIZE
Firm size is defined as the log of the price per
share times the number of shares outstanding.

In [ ]:
SIZE = np.log1p(price * num_share[price.columns])
SIZE = SIZE.resample("ME").last()

## DTURN
Detrended turnover is the six-month average of
monthly share turnover subtracted by the prior 18-
month average.

In [ ]:
TURN = (volume.resample('ME').sum() / num_share.resample('ME').last())

In [ ]:
DTURN = (TURN.rolling(6).mean() - TURN.rolling(18).mean()).dropna(how='all')

## AGE
Firm age is the number of years since the firm’s
first appearance on the CRSP monthly stock file.

In [ ]:
monthly_price = price.resample('ME').last()
AGE = monthly_price.copy()

# 각 열의 유효 데이터 순서를 바꾸는 코드
for col in AGE.columns:
    # 유효한 값(결측치 제외) 인덱스 추출
    valid_index = AGE[col].dropna().index
    # 순서대로 값 할당
    AGE.loc[valid_index, col] = range(1, len(valid_index) + 1)

## TANG (유형자산비율)
Tangible assets at the end of June in year t to
May in year t + 1 are property, plant, and equipment
(Compustat annual item PPEGT) divided by total assets (item AT) for the end of year t − 1.

총자산으로 나눠야댐

In [ ]:
TANG = (tangible / toa).shift(6).dropna(how='all') 
TANG = TANG.resample("ME").last()

## SALESG
Sales growth at the end of June in year t to
May in year t + 1 is the log difference between sales
(Compustat annual item SALE) at the end of year
t − 1 and sales at the end of year t − 2

In [ ]:
SALESG = (np.log1p(sales) - np.log1p(sales.shift(3))).shift(6)
SALESG = SALESG.resample("ME").last()

## DRETAIL
(개인순매수량/상장주식수) 변화

In [ ]:
DRETAIL = (retail.dropna(how='all').resample("ME").sum().replace(0, np.nan) / num_share.resample("ME").last().loc["2000":]) * 100

# DINIT = (init.dropna(how='all').resample("ME").sum().replace(0, np.nan) / num_share.resample("ME").last().loc["2000":]) * 100

## CIRG (Capital Impairment Ratio Growth)

자본잠식률 = (자본금-자본총계) / 자본금

In [ ]:
CAIP = ((capital - toe)/toe)
CAIPG = (np.log1p(CAIP) - np.log1p(CAIP.shift(3))).shift(6)

CAIPG = CAIPG.resample("ME").last().dropna(how="all")

# 폭락정의

In [ ]:
month = 6
threshold = 0.35 / 1

fwd_ret = (np.log1p(price.resample("ME").last().pct_change(periods=month, fill_method=None))).shift(-month)
crash = fwd_ret.apply(lambda x: np.where(x>=threshold, 1, np.where(x<=-threshold, -1, 0)))

# 모델 학습 (In-Sample)

In [ ]:
result = []
result_dict = {}
for name in tqdm(ticker):
    
    tmp_df = RM12.copy()
    tmp_df['EXRET12'] = EXRET12[name]
    tmp_df['TVOL'] = TVOL[name]
    tmp_df['TSKEW'] = TSKEW[name]
    tmp_df['SIZE'] = SIZE[name]
    tmp_df['DTURN'] = DTURN[name]
    tmp_df['AGE'] = AGE[name]
    tmp_df['TANG'] = TANG[name]
    tmp_df['SALESG'] = SALESG[name]
    tmp_df['DRETAIL'] = DRETAIL[name]
    # tmp_df['DINIT'] = DINIT[name]
    tmp_df["CAIPG"] = CAIPG[name]
    
    tmp_df['Crash'] = crash[name]

    tmp_df.dropna(inplace=True)
    result.append(tmp_df)
    
    result_dict[name] = tmp_df.rename(columns={"Crash":f"{name}", "코스피":"RM12"})
    
    
result_df = pd.concat(result).sort_index()
result_df = result_df.rename(columns={"코스피":"RM12"})
    

In [ ]:
result_df.corr()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1. 데이터 준비
X = result_df.loc[:"2020-01-01"].drop(columns=['Crash'])
y = result_df.loc[:"2020-01-01"]['Crash']

X_test = result_df.loc["2021-01-01":].drop(columns=['Crash'])
y_test = result_df.loc["2021-01-01":]['Crash']

# 2. 클래스 레이블 변환 (-1, 0, 1 → 0, 1, 2)
# 기준 클래스가 `0`이 되도록 순서 지정
y_transformed = y.map({0: 0, -1: 1, 1: 2})  # 기준 클래스를 0으로 설정

# 3. SMOTE를 적용하여 오버샘플링
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y_transformed)

# 4. statsmodels로 로지스틱 회귀
# 데이터에 상수항 추가 (절편 포함)
X_resampled_const = sm.add_constant(X_resampled)

# 다중 로지스틱 회귀 모델 생성 및 학습
sm_model = sm.MNLogit(y_resampled, X_resampled_const)
result = sm_model.fit()

# 5. 회귀 결과 요약 출력
print(result.summary())

# 6. 테스트 데이터에 대해 예측 (상수항 추가)
X_test_const = sm.add_constant(X_test)
y_pred_prob = result.predict(X_test_const)
y_pred_encoded = y_pred_prob.idxmax(axis=1)

# 7. 클래스 레이블 복원 (0, 1, 2 → -1, 0, 1)
inverse_mapping = {0: 0, 1: -1, 2: 1}
y_pred = y_pred_encoded.map(inverse_mapping)

# 8. 평가
from sklearn.metrics import classification_report, accuracy_score
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

In [ ]:
# 혼동 행렬 계산
cm = confusion_matrix(y_test, y_pred)
score = round(cm[0][0]/cm[2][0],3)

# 혼동 행렬 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[-1, 0, 1], yticklabels=[-1, 0, 1])
plt.title(f"{month}month / -{round(threshold*100)}% / Score: {score} \n{list(result_df.drop(columns=["RM12","EXRET12","TVOL","TSKEW","Crash","SIZE","DTURN","AGE","TANG","SALESG","DRETAIL"]).columns)}")
plt.xlabel("Predicted Labels")  # X축: 예측 값
plt.ylabel("Actual Labels")     # Y축: 실제 값
plt.show()

# 모델 학습 (Out of Sample)

In [ ]:
crashp_df = pd.DataFrame(columns=ticker, index=ret.index)

def process_month(idx, result_df, result_dict, ticker):
    
    """
    하나의 idx(월) 에 대해 다음을 수행:
      - X, y 생성
      - SMOTE 오버샘플링
      - 다중 로지스틱 회귀(MNLogit)
      - pred_idx(13개월 후 월 말)에 대해 예측
      - { 종목명: crash 확률 } 딕셔너리를 반환
    """
    
    # 1) 모델 학습용 데이터(X, y) 준비
    X = result_df.loc[:idx].drop(columns=['Crash'])
    y = result_df.loc[:idx]['Crash']
    
    # 2) 타겟 변환 (0 -> 0, -1 -> 1, 1 -> 2)
    y_transformed = y.map({0: 0, -1: 1, 1: 2})
    
    # 3) SMOTE 적용
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y_transformed)

    # 4) 다중 로지스틱 회귀 모델
    X_resampled_const = sm.add_constant(X_resampled)
    sm_model = sm.MNLogit(y_resampled, X_resampled_const)
    model = sm_model.fit(disp=False)

    # 5) 예측에 사용할 날짜 (7개월 후, 월 말)
    pred_idx = idx + pd.DateOffset(months=7)
    pred_idx = pred_idx + pd.offsets.MonthEnd(0)

    print(f"[{idx}] -> 예측 시점: [{pred_idx}]")

    # 6) 결과 저장용 딕셔너리 ( { 종목명: crash 확률 } )
    result_for_month = {}

    # 7) 각 종목별 예측
    for name in ticker:
        try:
            # 예측 데이터 준비
            X_test = (sm.add_constant(result_dict[name])).loc[[pred_idx]].drop(columns=name)

            # 폭락 확률 (index : 1)
            # 잭팟 확률 (index : 2)
            crashp = model.predict(X_test)[2].values[0]
            result_for_month[name] = crashp

            print(f" - {name} crash prob: {crashp}")

        except:
            # 데이터가 없거나 컬럼이 누락되었을 경우
            pass

    # pred_idx와 {종목명: 확률} 딕셔너리를 반환
    return pred_idx, result_for_month

# --- 병렬 처리 부분 --- #
# 월 단위 인덱스 리스트
month_list = sorted(set(result_df.loc["2005-01":].index))

# n_jobs=-1 은 CPU 코어를 모두 활용한다는 의미
results = Parallel(n_jobs=-1)(
    delayed(process_month)(idx, result_df, result_dict, ticker)
    for idx in tqdm(month_list)
)

# --- 결과 취합 --- #
for pred_idx, dict_per_month in results:
    for name, crash_val in dict_per_month.items():
        crashp_df.loc[pred_idx, name] = crash_val

# 이제 crashp_df에는 병렬로 계산된 폭락 확률들이 채워져 있게 됩니다.

In [ ]:
# crashp_df.dropna(how='all').to_csv('output/crashp(6m,35%).csv', encoding="CP949")

In [ ]:
crashp_df.dropna(how='all').to_csv('output/jackp(6m,35%).csv', encoding="CP949")

# 포트폴리오 구성

In [ ]:
crashp0 = pd.read_csv('output/crashp(6m,35%).csv', encoding="CP949", index_col=0, parse_dates=True)
crashp = crashp0.dropna(how='all')

In [ ]:
port_list = calculate_quantile_ranges(crashp.shift(1), ret, 10)

In [ ]:
crash_stock = port_list[-1].copy()

# 유효한 값을 칼럼명으로 대체
for col in crash_stock.columns:
    crash_stock[col] = crash_stock[col].apply(lambda x: col if pd.notnull(x) else np.nan)

# print("변환된 데이터프레임:")
# list(crash_stock.iloc[[-1]].dropna(axis=1).columns)

In [ ]:
num_of_stock = round(port_list[-1].count(axis=1).mean())
num_of_stock

In [ ]:
port = {}
bench = kospi.resample("ME").last().loc[crashp.index]

plt.figure(figsize=(16,7))

for i in range(len(port_list)):

    port[i] = ((port_list[i]).mean(axis=1))
    plt.plot(np.log1p(port[i]).cumsum(), label=f'Crash{i+1}')

plt.plot(np.log(bench/bench.iloc[0]), label='KOSPI')
plt.legend()
plt.title(f'Mean Constituents Number : {num_of_stock}')
plt.show()

In [ ]:
port = {}

# 결과를 저장할 리스트
bar_values = []
bar_labels = []

for i in range(len(port_list)):
    
    port[i] = ((port_list[i]).mean(axis=1))
    
    # log1p를 적용한 후 sum 계산
    bar_values.append(np.log1p(port[i]).sum())
    bar_labels.append(f'Crash{i+1}')

# bar plot 생성
plt.figure(figsize=(14, 7))
plt.bar(bar_labels, bar_values)
plt.ylabel("Log Cumulative Return")
plt.title(f"Constituents: {num_of_stock} / 6month / Crash: -35% / Num of")

# xtick 회전 추가
plt.xticks(rotation=90)  # 45도 회전
plt.show()

# 종목만 선택해서 내보내기

# 한칸 골라서

In [ ]:
selected_stock = port_list[-1].copy()

# 유효한 값을 칼럼명으로 대체
for col in selected_stock.columns:
    selected_stock[col] = selected_stock[col].apply(lambda x: col if pd.notnull(x) else np.nan)
selected_stock.dropna(axis=1, how='all', inplace=True)

# 각 행(row)별로 NaN이 아닌 값들을 왼쪽으로 이동
def shift_left(row):
    return [x for x in row if pd.notnull(x)] + [np.nan] * (len(row) - sum(pd.notnull(row)))

# 데이터프레임 변환
selected_stock = selected_stock.apply(shift_left, axis=1, result_type='broadcast')

# 새로운 칼럼명 설정
selected_stock.columns = [f"Crash_Top{i+1}" for i in range(selected_stock.shape[1])]

selected_stock.dropna(axis=1, how='all', inplace=True)
selected_stock

# SIZE 규제

In [ ]:
jackp = pd.read_csv('output/jackp(6m,35%).csv', encoding="CP949", index_col=0, parse_dates=True)

In [ ]:
num = 10
quantile_num_prob = 15

size_q = calculate_quantile_df(SIZE, num)

In [ ]:
tmp_list = []
size_crash = {}

for i in tqdm(range(len(size_q))):
    
    tmp_list.append(((size_q[i] / size_q[i]) * jackp).dropna(how='all')) # 1로 masking 해서 crashp 값 가지도록함함
    size_crash[i] = (calculate_quantile_df(tmp_list[i], quantile_num_prob)) 


In [ ]:
size_return = copy.deepcopy(size_crash)
winsored_ret = winsorize_dataframe_entire(ret, lower_percentile=0.0, upper_percentile=0.05)

for i in tqdm(range(num)):
    for j in range(quantile_num_prob):
        
        size_return[i][j] = ((size_return[i][j] / size_return[i][j]).shift(1) * winsored_ret).mean(axis=1).dropna(how='all')
        mean = size_return[i][j].mean()
        std = size_return[i][j].std()
        size_return[i][j] = mean / std

In [ ]:
# 데이터프레임 생성 및 시각화용 정리
heatmap_data = pd.DataFrame(size_return).T  # DataFrame 변환 (시각화에 적합하게 전치)

heatmap_data.index.name = "Size"          # 행 이름을 'Size'로 설정
heatmap_data.columns.name = "Quantile"    # 열 이름을 'Quantile'로 설정

# 히트맵 시각화
plt.figure(figsize=(20, 8))
sns.heatmap(heatmap_data*100, square=True,annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Mean Return'})

plt.title("Size Controlled Return")

plt.xlabel("Jackpot Prob")
plt.ylabel("Size")

plt.show()

In [ ]:
tmp_port = size_crash[0][8]
tmp_prob = pd.DataFrame(tmp_port.iloc[-2].dropna().sort_values(ascending=False))
tmp_prob

In [ ]:
tmp_port = size_crash[1][-1]

tmp_prob = pd.DataFrame(tmp_port.iloc[-1].dropna().sort_values(ascending=False))
tmp_prob

In [ ]:
import yfinance as yf
kospi = yf.download("^KS11", start=tmp_port.index[0], end=tmp_port.index[-1], interval='1mo')["Close"]
kospi= kospi / kospi.iloc[0]

In [ ]:
cumulative_port = (np.log1p(((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1)).cumsum())
((cumulative_port)).plot(figsize=(15,8))
((kospi)).plot(figsize=(15,8))

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tmp_port = tmp_port.loc["2010":]

# 스타일 설정
sns.set_style("white")
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 데이터 준비 및 계산 ---
# 1. KOSPI 데이터 로드 및 인덱스 정렬
kospi = yf.download("^KS11", start=tmp_port.index[0], end=tmp_port.index[-1], interval='1mo')["Close"]
if isinstance(kospi, pd.DataFrame): kospi = kospi.iloc[:, 0]
kospi.index = kospi.index.tz_localize(None)
kospi = kospi.reindex(tmp_port.index, method='ffill') # 포트폴리오 인덱스에 맞춤

# 2. 로그 누적 수익률 계산 (시작점 0으로 통일)
# 포트폴리오
port_log_ret = np.log1p(((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1)).fillna(0)
port_cum = port_log_ret.cumsum()
port_cum = port_cum - port_cum.iloc[0] # 0에서 시작

# KOSPI
kospi_log_ret = np.log(kospi / kospi.shift(1)).fillna(0)
kospi_cum = kospi_log_ret.cumsum()
kospi_cum = kospi_cum - kospi_cum.iloc[0] # 0에서 시작

# 3. MDD 계산을 위한 Wealth Index 변환
port_wealth = np.exp(port_cum)
kospi_wealth = np.exp(kospi_cum)

# MDD 계산
port_dd = (port_wealth / port_wealth.cummax()) - 1
kospi_dd = (kospi_wealth / kospi_wealth.cummax()) - 1

# 4. 히트맵용 월별 수익률 데이터
monthly_ret = port_wealth.pct_change().fillna(0)
monthly_ret_df = pd.DataFrame({'year': monthly_ret.index.year, 'month': monthly_ret.index.month, 'return': monthly_ret})
monthly_heatmap = monthly_ret_df.pivot(index='year', columns='month', values='return') * 100

In [ ]:
plt.figure(figsize=(8, 5), dpi=300)

# 플롯
plt.plot(port_cum.index, port_cum, label='Q-Square Strategy', color="#203bda", linewidth=2)
plt.plot(kospi_cum.index, kospi_cum, label='KOSPI Benchmark', color="#58585a", linestyle='--', alpha=0.6)

# 데코레이션
plt.title("누적 수익률 추이 (Cumulative Log Return)", fontsize=16, fontweight='bold', pad=15)

plt.ylabel("Return", fontsize=12)
#plt.axhline(0, color='black', linewidth=0.8) # 0선
plt.legend(fontsize=12, loc='best')


# 최종 수치 텍스트
final_val = port_cum.iloc[-1]
final_kospi = kospi_cum.iloc[-1]


plt.text(port_cum.index[-1], final_val, f" {final_val:.2f}", color='#203bda', fontweight='bold', va='center', fontsize=17)
plt.text(kospi_cum.index[-1], final_kospi, f" {final_kospi:.2f}", color="#110505", fontweight='bold', va='center',fontsize=12)

#plt.tight_layout()
plt.show()

In [ ]:
final_val

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 스타일 설정
sns.set_style("white")
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 1. 데이터 준비 (이동평균 계산) ---
# (port_dd, kospi_dd는 이미 정의되어 있다고 가정)

# 6개월 이동평균 (Moving Average) 계산
window = 3
port_dd_ma = port_dd.rolling(window=window).mean()
kospi_dd_ma = kospi_dd.rolling(window=window).mean()

plt.figure(figsize=(10, 5), dpi=300)

# --- 2. 그래프 그리기 ---

# [A] 내 포트폴리오
# 2. Moving Average (진하게: Main Line)
plt.plot(port_dd_ma.index, port_dd_ma, label=f'Portfolio ({window}M MA)', 
         color='#004c99', linewidth=2.5) # 더 진한 파랑
# 3. 채우기 (흐리게)
plt.fill_between(port_dd_ma.index, port_dd_ma, 0, color='#1f77b4', alpha=0.05)


# [B] KOSPI (벤치마크)
# 1. Raw Data (흐리게)
# 2. Moving Average (진하게)
plt.plot(kospi_dd_ma.index, kospi_dd_ma, label=f'KOSPI ({window}M MA)', 
         color='black', linestyle='--', linewidth=2, alpha=0.8)


# --- 3. Max MDD 지점 표시 (이전 로직 유지) ---
# 텍스트 배경 박스 스타일
bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="lightgray", alpha=0.9)


# --- 4. 레이아웃 및 데코레이션 ---
plt.title(f"Max Drawdown(%) ({window}개월 평균)", fontsize=16, fontweight='bold', pad=20)
plt.ylabel("Drawdown (%)", fontsize=12)
plt.axhline(0, color='grey', linewidth=1)

# 범례 정리 (너무 많으면 복잡하므로 2열로 배치)
plt.legend(loc='lower left', ncol=2, fontsize=15, frameon=True)
# ★ 핵심: 상단과 우측 테두리 제거 (Despine) ★
# left=False, bottom=False는 왼쪽(Y축)과 아래(X축) 선은 남긴다는 뜻
sns.despine(top=True, right=True, left=True, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 8)) # 히트맵은 정사각형 비율이 보기 좋음

# 히트맵 그리기
sns.heatmap(monthly_heatmap, 
            annot=True,          # 숫자 표시
            fmt=".1f"           ,# 소수점 1자리
            #square=True,       # 셀을 정사각형으로
            cmap="RdYlGn",       # 빨강(손실)-노랑-초록(수익)
            center=-3,            # 0을 기준으로 색상 분기
            # linewidths=0.5,      # 셀 간격
            #linecolor='white',
            cbar_kws={'label': '수익률 (%)'})

# 데코레이션
plt.title("월별 수익률 히트맵 (Monthly Returns %)", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Year", fontsize=12)
plt.yticks(rotation=0) # 연도 가로로 표시

plt.tight_layout()
plt.show()

In [ ]:
tmp_port = size_crash[1][-1]
tmp_ret = (((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1))
tmp_ret = tmp_ret.loc["2010":]
tmp_ret

In [ ]:
(tmp_ret - kospi.pct_change()).dropna()

In [ ]:
(tmp_ret - kospi.pct_change()).dropna()[(tmp_ret - kospi.pct_change()).dropna()>=0]

In [ ]:
108/188 # 코스피

In [ ]:
120/190

In [ ]:
tmp_ret[tmp_ret >= 0]

In [ ]:
np.log1p(((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1))

In [ ]:
pd.DataFrame(((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1)).describe()

In [ ]:
0.086642 * np.sqrt(12) * 100

In [ ]:
import numpy as np

# 누적 로그 수익률을 일반 수익률로 변환
cumulative_returns = np.expm1(cumulative_port.loc["2010":])

# 최고 수익점(peak) 갱신
running_max = cumulative_returns.cummax()

# 최대 낙폭 계산
drawdown = (cumulative_returns - running_max) / running_max
mdd = drawdown.min()

# MDD 발생한 날짜 찾기
mdd_index = drawdown.idxmin()

print(f"Maximum Drawdown (MDD): {mdd:.2%}")
print(f"MDD 발생 날짜: {mdd_index}")



In [ ]:
np.expm1(cumulative_port[-1] / (len(cumulative_port) / 12)) * 100

In [ ]:
tmp = ((tmp_port / tmp_port).shift(1) * ret).dropna(how='all').mean(axis=1)
tmp.reset_index(drop=True).reset_index().corr()